In [2]:
import os, time, re, html
from io import BytesIO
from datetime import datetime

import requests
import pandas as pd
import yaml
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("SWISSDOX_API_KEY")
API_SECRET = os.getenv("SWISSDOX_API_SECRET")
if not API_KEY or not API_SECRET:
    raise RuntimeError("Swissdox API keys missing. Set SWISSDOX_API_KEY and SWISSDOX_API_SECRET in .env")

BASE = "https://swissdox.linguistik.uzh.ch/api"
URL_QUERY = f"{BASE}/query"
URL_STATUS = f"{BASE}/status"
HEADERS = {"X-API-Key": API_KEY, "X-API-Secret": API_SECRET}

QUERY_BASE_NAME = "BuerokratieVerwaltung_2025"
QUERY_NAME = f"{QUERY_BASE_NAME}_{datetime.now():%Y%m%d_%H%M%S}"
QUERY_COMMENT = "Requête générée depuis VS Code"
EXPIRATION_DATE = "2026-01-30"

START_DATE = "2025-01-01"
END_DATE = "2025-12-31"
LANGUAGES = ["de", "fr"]
SOURCES = ["NZZO","NNTA","NNHEU","ZWSO","TPS","NZZ","TA","ZWAO","TPSO","HEU","ZWAS","NZZS","ZWAI"]
MAX_RESULTS = 20000


def clean_text(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = html.unescape(s).replace("\r", " ").replace("\n", " ").replace("\t", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s.strip(' "“”„\'')


def clean_xml_swissdox(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = html.unescape(s)
    s = re.sub(r"</p>", "\n", s)
    s = re.sub(r"<[^>]+>", " ", s)
    return re.sub(r"\s+", " ", s).strip()


DE_TERMS = ["Bürokratie","Berner Verwaltung","Papierkrieg","Verwaltung","Bundesverwaltung","Beamtenapparat","Amtsschimmel","Regulierungsdichte","Behörden","Bürokraten","Beamte","Staatsangestellte"]
DE_LEVEL = ["Bund", "Bundes", "Kanton", "kantonal", "Schweiz"]
FR_TERMS = ["Bureaucratie","Administration publique","Administration fédérale","Appareil administratif","Appareil étatique","Appareil de l’État","Autorités administratives","Services de l’État","Services publics","Fonction publique","Pouvoir administratif","Autorités cantonales","Administration centrale","Départements fédéraux","Offices fédéraux","Organes de l’État","Technocratie","Bureaucrates","Fonctionnaires","Employés de l'État"]
FR_LEVEL = ["fédéral", "federal", "federale", "cantonal", "cantonale", "Suisse"]
DEPARTMENTS = [
    "VBS","DDPS","Eidgenössische Departement für Verteidigung, Bevölkerungsschutz und Sport","Département fédéral de la défense, de la protection de la population et des sports",
    "EDA","DFAE","Eidgenössische Departement für auswärtige Angelegenheiten","Département fédéral des affaires étrangères",
    "UVEK","DETEC","Eidgenössische Departement für Umwelt, Verkehr, Energie und Kommunikation","Département fédéral de l'environnement, des transports, de l'énergie et de la communication",
    "EJPD","DFJP","Eidgenössische Justiz- und Polizeidepartement","Département fédéral de justice et police",
    "EDI","DFI","Eidgenössische Departement des Innern","Département fédéral de l'intérieur",
    "EFD","DFF","Eidgenössische Finanzdepartement","Département fédéral des finances",
    "WBF","DEFR","Eidgenössische Departement für Wirtschaft, Bildung und Forschung","Département fédéral de l'économie, de la formation et de la recherche",
]

query_block = {
    "sources": SOURCES,
    "dates": [{"from": START_DATE, "to": END_DATE}],
    "languages": LANGUAGES,
    "content": {"OR": [{"AND": [{"OR": DE_TERMS}, {"OR": DE_LEVEL}]}, {"AND": [{"OR": FR_TERMS}, {"OR": FR_LEVEL}]}, {"OR": DEPARTMENTS}]},
}

yaml_payload = {
    "query": query_block,
    "result": {
        "format": "TSV",
        "maxResults": MAX_RESULTS,
        "columns": [
            "id","pubtime","medium_code","medium_name","rubric","regional",
            "doctype","doctype_description","language","char_count","dateline",
            "head","subhead","content_id","content",
        ],
    },
    "version": "1.2",
}

yaml_query = yaml.safe_dump(yaml_payload, sort_keys=False, allow_unicode=True)

payload = {
    "query": yaml_query,
    "name": QUERY_NAME,
    "comment": QUERY_COMMENT,
    "expirationDate": EXPIRATION_DATE,
}

r = requests.post(URL_QUERY, headers=HEADERS, data=payload)
if r.status_code >= 400:
    print("❌ Swissdox /query failed")
    print("status:", r.status_code)
    print("response headers:", dict(r.headers))
    print("response body (first 2000 chars):")
    print(r.text[:2000])
    print("\nSent headers:", dict(r.request.headers))
    body = r.request.body
    if isinstance(body, (bytes, bytearray)):
        body = body[:800].decode("utf-8", errors="replace")
    else:
        body = str(body)[:800]
    print("\nSent body (first 800 chars):")
    print(body)

r.raise_for_status()
resp = r.json()
if resp.get("result") != "ok":
    raise SystemExit(f"❌ Swissdox non-ok: {resp}")

query_id = resp.get("queryId") or resp.get("id")
if not query_id:
    raise SystemExit(f"❌ queryId introuvable: {resp}")
print(f"✅ queryId = {query_id}")

# Poll /status until downloadUrl
for _ in range(300):
    rs = requests.get(URL_STATUS, headers=HEADERS)
    rs.raise_for_status()
    job = next((j for j in rs.json() if j.get("id") == query_id), None)
    if not job:
        time.sleep(5)
        continue
    if job.get("error"):
        raise SystemExit(f"❌ Swissdox error: {job['error']}")
    download_url = job.get("downloadUrl")
    if download_url:
        break
    time.sleep(5)
else:
    download_url = None

if not download_url:
    raise SystemExit("❌ downloadUrl manquant (pas de fichier).")

if download_url.startswith("http"):
    download_full_url = download_url
elif download_url.startswith("/"):
    download_full_url = f"{BASE}{download_url}"
else:
    download_full_url = f"{BASE}/download/{download_url}"

print("⬇️ Download:", download_full_url)
r_dl = requests.get(download_full_url, headers=HEADERS)
r_dl.raise_for_status()

df_raw = pd.read_csv(BytesIO(r_dl.content), sep="\t", compression="xz")
print("✅ df_raw:", df_raw.shape)

# Clean df_raw -> df_articles

df_articles = df_raw.copy()
if "pubtime" in df_articles.columns:
    df_articles["pubtime"] = pd.to_datetime(df_articles["pubtime"].astype(str), errors="coerce", utc=True).dt.date

for c in ["medium_name", "rubric", "dateline", "head", "subhead"]:
    if c in df_articles.columns:
        df_articles[c] = df_articles[c].apply(clean_text)

if "content" in df_articles.columns:
    df_articles["content"] = df_articles["content"].apply(clean_xml_swissdox).apply(clean_text)

print("✅ df_articles (clean):", df_articles.shape)


✅ queryId = 88612662-34a9-47d3-85ed-9799923da8c0
⬇️ Download: https://swissdox.linguistik.uzh.ch/api/download/88612662-34a9-47d3-85ed-9799923da8c0__2026_01_10T20_29_25.tsv.xz
✅ df_raw: (8784, 15)
✅ df_articles (clean): (8784, 15)
